In [ ]:
import os
import random
import operator
import sys
from collections import Counter
import collections
import math
import numpy as np
import gensim
from util import *

## 一些功能函数
需要阅读这些函数来对实验中用到的数据结构，步骤有一些理解

In [ ]:
def dotProduct(d1, d2):
    """
    @param dict d1: a feature vector represented by a mapping from a feature (string) to a weight (float).
    @param dict d2: same as d1
    @return float: the dot product between d1 and d2
    """
    if len(d1) < len(d2):
        return dotProduct(d2, d1)
    else:
        return sum(d1.get(f, 0) * v for f, v in d2.items())


In [ ]:
def increment(d1, scale, d2):
    """
    Implements d1 += scale * d2 for sparse vectors.
    @param dict d1: the feature vector which is mutated.
    @param float scale
    @param dict d2: a feature vector.
    """
    for f, v in d2.items():
        d1[f] = d1.get(f, 0) + v * scale


In [ ]:
def readExamples(path):
    """
    读取数据
    """
    examples = []
    for line in open(path, encoding="ISO-8859-1"):
        # Format of each line: <output label (+1 or -1)> <input sentence>
        y, x = line.split(" ", 1)
        examples.append((x.strip(), int(y)))
    print("Read %d examples from %s" % (len(examples), path))
    return examples


In [ ]:
def evaluatePredictor(examples, predictor):
    """
    在|examples|上测试|predictor|的性能，计算正确率，返回正确率
    """
    error = 0
    for x, y in examples:
        if predictor(x) != y:
            error += 1
    error_rate = 1.0 * error / len(examples)
    accuracy = 1 - error_rate

    return accuracy

## Feature extrator (Your codes here)

### (1) 使用BOW作为特征
(a) 复习BOW
(b) 如何把一个句子（字符串）转化成BOW的特征？

In [ ]:
def extractFeatures_bow(x):
    """
    从字符串x中提取特征
    @param string x:
    @return dict: a feature vector represented by a mapping from a feature (string) to a weight (float).
    """
    # BEGIN_YOUR_CODE
    word_features = dict()
    for i in x.split(" "):
        if i not in word_features:
            word_features[i] = 1
        else:
            word_features[i] += 1
    return word_features
    # END_YOUR_CODE

### (2) 使用N-Gram作为特征
(a) 复习N-Gram相关的内容
(b) 字级别N-Gram还是词语级别？
(c) 如何把一个句子（字符串）转化成N-Gram的特征？

In [ ]:
def extractFeatures_ngram(x):
    """
    从字符串x中提取特征
    @param string x:
    @return dict: a feature vector represented by a mapping from a feature (string) to a weight (float).
    """
    # BEGIN_YOUR_CODE
    n = 7
    counts = collections.Counter()
    string = "".join(x.split())
    for i in range(0, len(string) - n + 1):
        counts[string[i : i + n]] += 1
    return counts
    # END_YOUR_CODE

### （3） 使用word2vec作为特征
(a) 获得词向量。你可以借鉴第一次的作业，使用gensim来自己训练一个word2vec, 或者加载预训练过的word2vec(gensim的网站上有说明如何下载并使用预训练过的词向量)。
(b) 考虑如何使用词向量得到句子的表示向量（feature）。
(c) 将向量转化为其余部分可以处理的形式（如：dict）
(d) 考虑如何**更好地**使用词向量得到句子的表示向量（feature）。

提示: 在ipynb的代码块中可以使用! 来执行命令行中的命令, 例如
```
!pip install gensim
```

In [ ]:
w2v = gensim.models.word2vec.load("embedding/word2vec_gensim")


def extractFeatures_word2vector(x):
    """
    从字符串x中提取特征
    @param string x:
    @return dict: a feature vector represented by a mapping from a feature (string) to a weight (float).
    """
    # BEGIN_YOUR_CODE
    words = list(gensim.utils.tokenize(x.lower()))
    # emb = np.zeros(200)
    emb = np.zeros(w2v.vector_size)
    cnt = 0
    for w in words:
        if w in w2v:
            # print(w)
            emb += w2v[w]
            cnt += 1
    emb /= len(words)
    # print(cnt / len(words))
    feat = {i: v for i, v in enumerate(emb)}
    # print(feat)
    return feat
    # END_YOUR_CODE

### (4) Test your feature extractor
实现了特征提取函数之后，可以简单地测试输出的正确性

In [ ]:
extractFeatures("a truly wonderful tale combined with stunning animation .")

## 学习与梯度更新
你需要理解题目中的loss_function, 自行推导出weights的更新公式，
通过对训练集上样本的迭代，来更新weights

In [ ]:
def learnPredictor(trainExamples, testExamples, featureExtractor, numIters, eta):
    """
    给定训练数据和测试数据，特征提取器|featureExtractor|、训练轮数|numIters|和学习率|eta|，
    返回学习后的权重weights
    你需要实现随机梯度下降优化权重
    """
    weights = {}
    for i in range(0, numIters):
        # BEGIN_YOUR_CODE
        totalloss = 0
        for sample in trainExamples:
            feature = featureExtractor(sample[0])
            wphix = 0
            for key in feature:
                if key not in weights:
                    weights[key] = 0
                wphix += weights[key] * feature[key]

            # calulatedLoss = 1 - wphix * sample[1]
            loss = max(0, (1 - wphix * sample[1]))
            totalloss += loss
            if loss > 0:
                # grad = {}
                for key in feature:
                    gradKey = feature[key] * sample[1] * -1
                    # grad[key] = gradKey
                    weights[key] -= eta * gradKey
        # END_YOUR_CODE
        trainAcc = evaluatePredictor(
            trainExamples,
            lambda x: (1 if dotProduct(featureExtractor(x), weights) >= 0 else -1),
        )
        testAcc = evaluatePredictor(
            testExamples,
            lambda x: (1 if dotProduct(featureExtractor(x), weights) >= 0 else -1),
        )
        print(
            "At iteration %d, Accuracy rate on training set is  %f, Accuracy rate on test set is %f "
            % (i, trainAcc, testAcc)
        )
    return weights


## 已经定义好的训练测试流程

In [ ]:
def TestModel(numIters, eta):
    trainExamples = readExamples("data/data_rt.train")
    testExamples = readExamples("data/data_rt.test")
    featureExtractor = extractFeatures_bow
    weights = learnPredictor(
        trainExamples, testExamples, featureExtractor, numIters=numIters, eta=eta
    )
    trainAcc = evaluatePredictor(
        trainExamples,
        lambda x: (1 if dotProduct(featureExtractor(x), weights) >= 0 else -1),
    )
    testAcc = evaluatePredictor(
        testExamples,
        lambda x: (1 if dotProduct(featureExtractor(x), weights) >= 0 else -1),
    )
    print("train accuracy = %s, test accuracy = %s" % (trainAcc, testAcc))


## 测试你的模型!
(a) 超参请自行更改
(b) 自行增加代码来进行训练loss和测试loss变化图的绘制
(b) 分析性能,模型泛化能力, 权重weights的可解释性等等

In [ ]:
TestModel(20, 0.01)